# Standard RAG with LangGraph

This notebook answers questions over the document we ingested in `4_ocr_chunk_store_pgvector.ipynb`. It is a **standard RAG** (Retrieval-Augmented Generation) workflow with two steps, wired together as a **LangGraph**:

```
question --> [ retrieve ] --> [ generate ] --> answer
                 |                  ^
          pgvector search      LLM + context
```

1. **Retrieve** - embed the question and pull the most similar chunks from the `rag_documents` collection in pgvector.
2. **Generate** - give those chunks to an LLM as context and ask it to answer using only that context.

Modelling it as a graph (rather than a single function) makes each step inspectable and easy to extend later - add a reranker, a query rewriter, or a check that loops back to retrieve more.

Prerequisite: run `4_ocr_chunk_store_pgvector.ipynb` first so the `rag_documents` collection is populated, and keep the pgvector database reachable (see `1_install_pgvector`).

## Dependencies

In addition to the ingestion packages, this notebook uses `langchain-openai` (the LLM client) and `langgraph` (the graph). All are in `../requirements.txt`:

```bash
pip install -r requirements.txt
```

## Configuration

Reuse the same `var.env` as the ingestion notebook for the pgvector connection. The LLM now points at the **self-hosted Qwen3.5-9B** vLLM service (see `day1skk/1a_install_llm`); the defaults below reach it via in-cluster DNS, so you only need to override them to call the model from outside the cluster.

```dotenv
# pgvector (same as notebook 4)
PG_HOST=pgvector       # in-cluster service name (see 05-service.yaml)
PG_PORT=5432
PG_USER=raguser
PG_PASSWORD=change-me-please
PG_DB=ragdb

# LLM (self-hosted Qwen3.5-9B on vLLM, OpenAI-compatible)
LLM_MODEL=Qwen/Qwen3.5-9B
LLM_BASE_URL=http://qwen-35-9b.default.svc.cluster.local/v1   # or http://<LoadBalancer-IP>/v1 from outside
LLM_API_KEY=sk-...      # the vLLM --api-key from 3_deployment.yaml
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("var.env")

PG_HOST = os.environ.get("PG_HOST", "pgvector")   # in-cluster service DNS name
PG_PORT = os.environ.get("PG_PORT", "5432")
PG_USER = os.environ.get("PG_USER", "raguser")
PG_PASSWORD = os.environ.get("PG_PASSWORD", "change-me-please")
PG_DB = os.environ.get("PG_DB", "ragdb")

CONNECTION = f"postgresql+psycopg://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"

# Must match the collection name used in the ingestion notebook.
COLLECTION_NAME = "rag_documents"

# Embedding model: the self-hosted Qwen3-Embedding-4B served by vLLM (see
# day2skk/0_install_emb_model). MUST match the model used at ingestion (notebook 4),
# or the question vector would not live in the same space as the stored chunks.
EMB_MODEL = os.environ.get("EMB_MODEL", "Qwen/Qwen3-Embedding-4B")
EMB_BASE_URL = os.environ.get("EMB_BASE_URL", "http://qwen3-emb-4b.default.svc.cluster.local/v1")
EMB_API_KEY = os.environ.get("EMB_API_KEY", "sk-dhU11z5FIjXQ8TLXpwDotpGetP21CQv3")

# LLM: the self-hosted Qwen3.5-9B served by vLLM (see day1skk/1a_install_llm).
# Defaults to the in-cluster service DNS; if you run this notebook OUTSIDE the
# cluster, set LLM_BASE_URL to the LoadBalancer IP, e.g. http://103.125.91.43/v1
LLM_MODEL = os.environ.get("LLM_MODEL", "Qwen/Qwen3.5-9B")
LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "http://qwen-35-9b.default.svc.cluster.local/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "sk-dhU11z5FIjXQ8TLXpwDotpGetP21CQv3")

print("pgvector:", f"postgresql+psycopg://{PG_USER}:***@{PG_HOST}:{PG_PORT}/{PG_DB}")
print("collection:", COLLECTION_NAME)
print("embeddings:", EMB_MODEL, "@", EMB_BASE_URL)
print("LLM:", LLM_MODEL, "@", LLM_BASE_URL)

## Connect to the vector store

We open the **same** `rag_documents` collection with the **same** embedding model used at ingestion time. The embedding model has to match, or the question vector would not live in the same space as the stored chunk vectors and search would be meaningless.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

# Must be the SAME embedding model used at ingestion (notebook 4): the self-hosted
# Qwen3-Embedding-4B on vLLM. check_embedding_ctx_length=False sends raw text.
embeddings = OpenAIEmbeddings(
    model=EMB_MODEL,
    base_url=EMB_BASE_URL,
    api_key=EMB_API_KEY,
    check_embedding_ctx_length=False,
)

vector_store = PGVector(
    embeddings=embeddings,
    collection_name=COLLECTION_NAME,
    connection=CONNECTION,
    use_jsonb=True,
)

# Quick check that the collection has content.
probe = vector_store.similarity_search("test", k=1)
print("retrieved a chunk:", bool(probe))

## The LLM

Generation uses the **self-hosted Qwen3.5-9B** model we deployed with vLLM in `day1skk/1a_install_llm`. vLLM exposes an OpenAI-compatible API, so we point `ChatOpenAI` at its `base_url` and pass the vLLM `--api-key`. Because this notebook runs inside the cluster, it reaches the model at the in-cluster service DNS `qwen-35-9b.default.svc.cluster.local` (set `LLM_BASE_URL` to the LoadBalancer IP to call it from outside).

In [ ]:
from langchain_openai import ChatOpenAI

# Points at the self-hosted Qwen3.5-9B vLLM endpoint (OpenAI-compatible API).
llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    temperature=0,
)

## Define the graph state and nodes

The shared **state** carries the `question`, the retrieved `context` (a list of documents), and the final `answer`.

- **`retrieve`** embeds the question and fetches the top-`k` chunks from pgvector.
- **`generate`** stuffs those chunks into the prompt and asks the LLM to answer using only that context (a guard against the model making things up).

In [ ]:
from typing import TypedDict, List
from langchain_core.documents import Document


class RAGState(TypedDict):
    question: str
    context: List[Document]
    answer: str


def retrieve(state: RAGState) -> dict:
    docs = vector_store.similarity_search(state["question"], k=4)
    print(f"[retrieve] found {len(docs)} chunks")
    return {"context": docs}


def generate(state: RAGState) -> dict:
    context_text = "\n\n".join(d.page_content for d in state["context"])
    messages = [
        ("system",
         "You are a helpful assistant. Answer the question using ONLY the context below. "
         "If the answer is not in the context, say you don't know."),
        ("human", f"Context:\n{context_text}\n\nQuestion: {state['question']}"),
    ]
    answer = llm.invoke(messages).content
    return {"answer": answer}

## Build the graph

The flow is a straight line: `START -> retrieve -> generate -> END`. `.compile()` turns it into a runnable app.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

rag = builder.compile()
print(rag.get_graph().draw_mermaid())

## Ask a question

`invoke` runs retrieve then generate and returns the full state: the answer plus the context that produced it. Showing the sources is what makes the answer *grounded* - you can trace every claim back to a retrieved chunk.

In [ ]:
result = rag.invoke({"question": "What is this document about?"})

print("ANSWER:\n", result["answer"])
print("\nSOURCES:")
for d in result["context"]:
    print(f"  - {d.metadata.get('source')} (chunk {d.metadata.get('chunk_index')})")

### Try your own question

Change the question and re-run. Because generation is constrained to the retrieved context, asking about something not in the document should make the model say it does not know - the intended behaviour for a grounded RAG system.

In [ ]:
question = "Summarize the key points in one sentence."
print(rag.invoke({"question": question})["answer"])

## Recap

- **Standard RAG = retrieve + generate.** Retrieval finds relevant chunks; generation answers using them as context.
- The retriever reads the **same pgvector collection** (`rag_documents`) with the **same embedding model** used during ingestion - both must match.
- Constraining the LLM to "answer only from the context" is what keeps answers **grounded** and reduces hallucination.
- Expressing it as a **LangGraph** (`retrieve -> generate`) keeps each step separate and inspectable, and leaves room to grow: add query rewriting, reranking, or a self-check loop.

This closes the standard RAG pipeline: **install pgvector -> upload/download the PDF -> OCR + chunk + store -> retrieve + generate**.